## tl;dr

Screening status: **5563 machine-screened**, **12 fact-check required**, and **71 rejected**. Public-safe export contains no question or answer text.

## Context & Methods

### Key Assumptions

The intended grain is one candidate span at one source position. Machine screening is conservative and does not replace human review.

In [1]:
from pathlib import Path
from collections import Counter
import csv, json
root = Path.cwd()
rows = [json.loads(line) for line in (root/'data/reviewed/questions.jsonl').read_text().splitlines() if line]
public = [json.loads(line) for line in (root/'data/exports/questions_public_safe.jsonl').read_text().splitlines() if line]
duplicates = list(csv.DictReader((root/'data/exports/duplicate_clusters.csv').open(encoding='utf-8-sig')))
print('rows / columns:', len(rows), len(rows[0]))
print('unique question ids:', len({row['question_id'] for row in rows}))
print('statuses:', dict(Counter(row['human_review_status'] for row in rows)))
print('duplicate member rows:', len(duplicates))

rows / columns: 5646 29
unique question ids: 5646
statuses: {'machine_screened': 5563, 'rejected': 71, 'fact_check_required': 12}
duplicate member rows: 1493


## Data

Checks cover completeness, uniqueness, enum validity, extraction confidence, review status, and public-safe leakage.

In [2]:
required = ['question_id','source_id','source_url','language','normalized_text','extraction_confidence']
print('required-field nulls:', {field: sum(not row.get(field) for row in rows) for field in required})
print('out-of-range confidence:', sum(not 0 <= row['extraction_confidence'] <= 1 for row in rows))
print('public-safe text leaks:', sum(any(row.get(field) for field in ['raw_text','normalized_text','answer_text']) for row in public))

required-field nulls: {'question_id': 0, 'source_id': 0, 'source_url': 0, 'language': 0, 'normalized_text': 0, 'extraction_confidence': 0}
out-of-range confidence: 0
public-safe text leaks: 0


## Results

Required fields and stable IDs are complete, but extraction quality is partial: PDF character-spacing damage, fragments, and marketing headings are present. Japanese candidates are present but remain a small minority.

## Takeaways

1. Human-review the machine-screened queue.
2. Repair multi-line PDF reconstruction before expanding volume.
3. Add a rights-reviewed Japanese public source.
4. Keep public-safe leakage tests blocking every export.